# Error Analysis


## Setup & Imports

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROMPTS_DIR = ROOT / "models" / "llama" / "prompts"         

EXPERIMENT_NAME = "experiment_test_9385_20260609_1013" 
EXPERIMENT_DIR  = ROOT / "data" / "experiments" / EXPERIMENT_NAME
LLAMA_RUNS  = EXPERIMENT_DIR / "llama_runs"
EXPERIMENT  = EXPERIMENT_DIR / "experiment.jsonl"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
print(f"ROOT = {ROOT}")

# --- Cache freshness: experiment.jsonl can change underneath us (e.g. pipeline §8.0 collapses it to
# the matched intersection I'). The retrieval caches below are rebuilt `if not exists`, so a stale
# copy from a larger run would silently be reused. Auto-invalidate any whose row count no longer
# matches experiment.jsonl; they rebuild in §1/§2.
_exp_rows = sum(1 for _ in open(EXPERIMENT, encoding="utf-8"))
print(f"experiment.jsonl rows = {_exp_rows}")
for _name in ("error_analysis_meta.jsonl", "error_analysis_relations.jsonl"):
    _cache = EXPERIMENT_DIR / _name
    if _cache.exists() and sum(1 for _ in open(_cache, encoding="utf-8")) != _exp_rows:
        _cache.unlink()
        print(f"  stale cache removed (will rebuild on {_exp_rows} rows): {_name}")

# similarities.npz is produced by the MAIN pipeline §8.1, not here -- warn loudly if it is stale.
_sim = EXPERIMENT_DIR / "similarities.npz"
if _sim.exists():
    import numpy as _np
    with _np.load(_sim) as _z:
        _shape0 = _z[next(iter(_z.files))].shape[0]
    if _shape0 != _exp_rows:
        print(f"  !! similarities.npz is {_shape0}x{_shape0} but experiment.jsonl has {_exp_rows} rows -- "
              f"re-run pipeline §8.1 on this experiment to regenerate it BEFORE the retrieval analyses (§1, §2).")

ROOT = /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis
experiment.jsonl rows = 4315


### Analysis 1 — Structural confusion: do wrong retrievals share a plot skeleton?

**Goal:** When the structural model retrieves the wrong story, is it because of a random error/noise, or whether the errors are meaningful where these "wrong" stories are structurally similar (e.g., share same archetypical event trigger and/or event type skeleton)? 

**Method:**
1. Take every query whose top-1 retrieval belongs to a different work, so the actual errors hapenned. Call the wrongly retrieved summary "wrong match". 
2. For each (query, wrong match) pair, measure how much narrative structure the two share, in four ways: 
    1. Shared *event types* via Traversky Index overlap: How much *event types* overlap between query and the wrong match?
    2. Longest in-order common event type sequence (namely, LCS ratio) for *event types*: What is the longest sequence of *event types* that appear in both the query and the wrong match? 
    3. Shared *event triggers* via Traversky Index overlap: How much *event triggers* overlap between query and the wrong match?
    4. Longest in-order common event type sequence (namely, LCS ratio) for *event triggers*: What is the longest sequence of *event triggers* that appear in both the query and the wrong match?
3. Build a chance baseline: pair the same queries with a random story (also a different work) and compute the same four measures. These are the ratios created, as if we would do things randomly.
4. Compare error pairs vs random pairs (one-sided Mann–Whitney U): if the scores of match between (query, wrong match) is higher than the scores of match (query, random match), the errors are systematic structural confusion (i.e., signal)
5. Visually inspect the top pairs by hand: the shared event sequence (the candidate plot skeleton), genres, languages, and the two texts side by side.

**Results:**
- 77.5% of queries (4,393 / 5,666) retrieve a wrong story at rank 1 (Qwen3 × events_only).
- The wrong story shares **almost twice** the event categories with the query that a random story would (Dice 0.44 vs 0.23).
- The same holds for **order**: the shared in-order event sequence is ~2× longer than chance (0.20 vs 0.11) — so it's plot *shape*, not just shared ingredients.
- Even on **exact trigger verbs**, the wrong match is ~3× above chance (0.10 vs 0.04) — so the effect is not an artifact of coarse or noisy event-type labels.
- All four differences are significant at p ≈ 0. Sanity check: the random baseline for event-type overlap (0.23) reproduces the corpus-wide mean from pipeline §8.8 / Table 5 (0.237).
- Top confusions share interpretable skeletons across languages and genres — e.g. *Sending → Arriving → Giving → Warning* shared by a German doctor drama and an Italian mafia film with **zero** shared genres.

In [2]:
import random

import numpy as np
from IPython.display import display, Markdown
from scipy.stats import mannwhitneyu

# Set up encoder and condition. Although all conditions have the same set of events, the retrieval results, therefore the wrong match, changes across conditions. 
#   VIEW (embedder): qwen3_emb_0p6b | e5_mistral
#   COND (representation): events_only | temporal | causal | temporal_causal_independent | temporal_causal_joint | raw_text
VIEW, COND = "qwen3_emb_0p6b", "events_only"
SEED = 0
# Download the data from cache, instead of re-loading the full data in each run to save time
META_CACHE   = EXPERIMENT_DIR / "error_analysis_meta.jsonl"
# Get the similarities across all conditions and embedders
SIMILARITIES = EXPERIMENT_DIR / "similarities.npz"

# 1. Get per-row metadata only, since experiment.jsonl carries embeddings (~4.4 GB)
if not META_CACHE.exists():
    with open(EXPERIMENT, encoding="utf-8") as f, open(META_CACHE, "w", encoding="utf-8") as out:
        for line in f:
            r = json.loads(line)
            out.write(json.dumps({
                "wikidata_id": r["wikidata_id"], "summary_id": r["summary_id"],
                "lang": r.get("lang"), "genres": r.get("genres") or [],
                "triggers": [e["trigger"].lower() for e in r.get("events") or []],
                "types":    [e["event_type"]      for e in r.get("events") or []],
                "text": r.get("text", ""),
            }, ensure_ascii=False) + "\n")
meta  = [json.loads(l) for l in META_CACHE.read_text(encoding="utf-8").splitlines() if l.strip()]
works = np.array([m["wikidata_id"] for m in meta])
N     = len(meta)

# 2. Rank-1 neighbor per query (similarities.npz rows follow experiment.jsonl order; diagonal is -inf)
sim = np.load(SIMILARITIES)[f"{VIEW}__{COND}"]
assert sim.shape == (N, N), f"similarities {sim.shape} vs meta rows {N} — cache stale? delete {META_CACHE}"
rank1 = sim.argmax(axis=1)
wrong = np.flatnonzero(works[rank1] != works)
error_pairs = [(int(q), int(rank1[q])) for q in wrong]
print(f"{VIEW}__{COND}: wrong story at rank 1 for {len(wrong)}/{N} queries ({len(wrong)/N:.1%})")

# 3. Skeleton-overlap measures. Inventory = set-based Dice; order = LCS ratio with the same normalization (2·LCS/(|a|+|b|)), so inventory and order numbers are directly comparable.
def dice(a, b):
    A, B = set(a), set(b)
    return 2 * len(A & B) / (len(A) + len(B)) if (A or B) else 0.0

def lcs_table(a, b):
    L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            L[i + 1][j + 1] = L[i][j] + 1 if x == y else max(L[i][j + 1], L[i + 1][j])
    return L

def lcs_ratio(a, b):
    return 2 * lcs_table(a, b)[-1][-1] / (len(a) + len(b)) if (a or b) else 0.0

# Calculate the measures
MEASURES = {
    "Event-type inventory overlap": lambda u, v: dice(u["types"], v["types"]),
    "Event-type order overlap":     lambda u, v: lcs_ratio(u["types"], v["types"]),
    "Trigger inventory overlap":    lambda u, v: dice(u["triggers"], v["triggers"]),
    "Trigger order overlap":        lambda u, v: lcs_ratio(u["triggers"], v["triggers"]),
}

# 4. Random (Null) Baseline: same queries, but each paired with a random non-relevant candidate instead of the model's wrong choice, basically this is what overlap looks like when the pairing carries no signal
rng = random.Random(SEED)
null_pairs = []
for q, _ in error_pairs:
    j = rng.randrange(N)
    while j == q or works[j] == works[q]:
        j = rng.randrange(N)
    null_pairs.append((q, j))

# 5. Error vs null per measure
scores, table = {}, []
for name, fn in MEASURES.items():
    err = np.array([fn(meta[q], meta[c]) for q, c in error_pairs])
    nul = np.array([fn(meta[q], meta[c]) for q, c in null_pairs])
    _, p = mannwhitneyu(err, nul, alternative="greater")
    scores[name] = (err, nul)
    table.append({"Measure": name,
                  "Error pair mean":  f"{err.mean():.4f}",
                  "Random-pair mean": f"{nul.mean():.4f}",
                  "Δ":                f"{err.mean() - nul.mean():+.4f}",
                  "p":                "< .001" if p < .001 else f"{p:.3f}"})
    
# Display the table, and a note for my thesis and other readers.
display(pd.DataFrame(table))
display(Markdown(
    f"*Note.* An *error pair* is a query and the wrong story retrieved for it at rank 1 under "
    f"*{VIEW} × {COND}* (embedder × representation condition; set `VIEW`/`COND` above to analyse "
    f"another cell of the ablation grid). A *random pair* is the same query paired with a randomly "
    f"drawn summary of a different work. *Inventory overlap* ignores order: the Dice coefficient "
    f"(symmetric Tversky index, α=β=0.5) between the two summaries' sets of items. "
    f"*Order overlap* respects narrative order: the longest common subsequence between the two item "
    f"sequences, normalized as *2·LCS/(|a|+|b|)*, so it is directly comparable to inventory overlap. "
    f"Both are computed over MAVEN event types and over exact (lowercased) trigger words. "
    f"*p*: one-sided Mann–Whitney U, H1 = error pairs overlap more than random pairs."
))

qwen3_emb_0p6b__events_only: wrong story at rank 1 for 3382/4315 queries (78.4%)


,Measure,Error pair mean,Random-pair mean,Δ,p
0,Event-type inventory overlap,0.4319,0.2295,+0.2024,< .001
1,Event-type order overlap,0.2010,0.1116,+0.0894,< .001
2,Trigger inventory overlap,0.0980,0.0364,+0.0616,< .001
3,Trigger order overlap,0.0644,0.0259,+0.0385,< .001


*Note.* An *error pair* is a query and the wrong story retrieved for it at rank 1 under *qwen3_emb_0p6b × events_only* (embedder × representation condition; set `VIEW`/`COND` above to analyse another cell of the ablation grid). A *random pair* is the same query paired with a randomly drawn summary of a different work. *Inventory overlap* ignores order: the Dice coefficient (symmetric Tversky index, α=β=0.5) between the two summaries' sets of items. *Order overlap* respects narrative order: the longest common subsequence between the two item sequences, normalized as *2·LCS/(|a|+|b|)*, so it is directly comparable to inventory overlap. Both are computed over MAVEN event types and over exact (lowercased) trigger words. *p*: one-sided Mann–Whitney U, H1 = error pairs overlap more than random pairs.

### 1.2 Stratified robustness
**Goal:** is the structural-confusion effect general, or an artifact of short summaries? Queries are split into quartiles by their number of extracted events, and the same error-vs-random comparison is run within each quartile. Δ > 0 with a significant p in EVERY quartile (incl. the longest summaries) ⇒ the effect is general.

In [3]:
q_events = np.array([len(meta[q]["types"]) for q, _ in error_pairs])   # event count of each error query
edges    = np.quantile(q_events, [0, .25, .5, .75, 1.0])
qbin     = np.clip(np.searchsorted(edges, q_events, side="right") - 1, 0, 3)   # quartile index 0..3

rows = []
for b in range(4):
    idx = np.flatnonzero(qbin == b)
    lo, hi = int(q_events[idx].min()), int(q_events[idx].max())
    for name in MEASURES:
        err, nul = scores[name]            # full arrays, aligned with error_pairs
        e, n = err[idx], nul[idx]
        _, p = mannwhitneyu(e, n, alternative="greater")
        rows.append({"Quartile": f"Q{b+1} ({lo}–{hi} events)", "n": len(idx), "Measure": name,
                     "Error pair mean":  f"{e.mean():.4f}",
                     "Random-pair mean": f"{n.mean():.4f}",
                     "Δ":                f"{e.mean() - n.mean():+.4f}",
                     "p":                "< .001" if p < .001 else f"{p:.3f}"})
display(pd.DataFrame(rows).set_index(["Quartile", "n", "Measure"]))
display(Markdown(
    f"*Note.* Same error and random pairs as the table above ({VIEW} × {COND}), stratified into "
    f"quartiles by the **query's number of extracted events** (a length proxy that is also the unit "
    f"the overlap measures operate on). Because each random pair reuses its error pair's query, the "
    f"two groups are identical on the query side within every quartile. A positive Δ with significant "
    f"*p* in all four quartiles shows the structural-confusion effect is general across summary "
    f"lengths, not an artifact of short summaries (overlap *ratios* are mechanically larger for "
    f"short event sequences, but that inflation applies to error and random pairs alike)."
))

Error pair mean Random-pair mean        Δ       p
Quartile           n   Measure                                                                       
Q1 (5–10 events)   771 Event-type inventory overlap          0.3546           0.1254  +0.2292  < .001
                       Event-type order overlap              0.2248           0.0786  +0.1462  < .001
                       Trigger inventory overlap             0.0836           0.0167  +0.0669  < .001
                       Trigger order overlap                 0.0717           0.0147  +0.0570  < .001
Q2 (11–26 events)  903 Event-type inventory overlap          0.3770           0.1972  +0.1798  < .001
                       Event-type order overlap              0.1872           0.1083  +0.0788  < .001
                       Trigger inventory overlap             0.0844           0.0286  +0.0558  < .001
                       Trigger order overlap                 0.0620           0.0229  +0.0391  < .001
Q3 (27–51 events)  854 Event-type inventory overlap          0.4619           0.2714  +0.1905  < .001
                       Event-type order overlap              0.1926           0.1282  +0.0644  < .001
                       Trigger inventory overlap             0.1008           0.0447  +0.0562  < .001
                       Trigger order overlap                 0.0607           0.0318  +0.0289  < .001
Q4 (52–126 events) 854 Event-type inventory overlap          0.5296           0.3156  +0.2140  < .001
                       Event-type order overlap              0.2027           0.1283  +0.0744  < .001
                       Trigger inventory overlap             0.1224           0.0540  +0.0684  < .001
                       Trigger order overlap                 0.0641           0.0334  +0.0307  < .001

*Note.* Same error and random pairs as the table above (qwen3_emb_0p6b × events_only), stratified into quartiles by the **query's number of extracted events** (a length proxy that is also the unit the overlap measures operate on). Because each random pair reuses its error pair's query, the two groups are identical on the query side within every quartile. A positive Δ with significant *p* in all four quartiles shows the structural-confusion effect is general across summary lengths, not an artifact of short summaries (overlap *ratios* are mechanically larger for short event sequences, but that inflation applies to error and random pairs alike).

### 1.3 Visually check the cases

In [4]:
from IPython.display import display, Markdown

TOP_K   = 3
PREVIEW = None   # chars of raw text shown per summary; None = full text (for the thesis appendix)

def cut(s):
    return s if PREVIEW is None or len(s) <= PREVIEW else s[:PREVIEW] + "…"

def lcs_seq(a, b):
    """Backtrack the DP table to recover one LCS (the shared skeleton itself)."""
    L = lcs_table(a, b)
    i, j, out = len(a), len(b), []
    while i and j:
        if a[i-1] == b[j-1]:
            out.append(a[i-1]); i -= 1; j -= 1
        elif L[i-1][j] >= L[i][j-1]:
            i -= 1
        else:
            j -= 1
    return out[::-1]

err_lcs = scores["Event-type order overlap"][0]
order   = np.argsort(-err_lcs)

md = [f"## Top {TOP_K} structural confusions under `{VIEW}__{COND}` (by event-type order overlap)\n"]
for rank, r in enumerate(order[:TOP_K], 1):
    q, c = error_pairs[r]
    mq, mc = meta[q], meta[c]
    shared_skel   = lcs_seq(mq["types"], mc["types"])
    shared_genres = sorted(set(mq["genres"]) & set(mc["genres"]))
    md.append(
        f"### {rank}. `{mq['wikidata_id']}__{mq['summary_id']}` → `{mc['wikidata_id']}__{mc['summary_id']}`"
        f" &nbsp; (cosine similarity {sim[q, c]:.3f}, event type order overlap {err_lcs[r]:.3f}, "
        f"event type inventory {dice(mq['types'], mc['types']):.3f}, event trigger inventory overlap {dice(mq['triggers'], mc['triggers']):.3f})\n\n"
        f"- **shared event-type skeleton ({len(shared_skel)} types)**: {' → '.join(shared_skel)}\n"
        f"- **shared genres**: {', '.join(shared_genres) if shared_genres else '(none)'}"
        f" &nbsp;|&nbsp; langs: {mq['lang']} vs {mc['lang']}\n\n"
        f"- **Event Types of the Query ({len(mq['types'])})**: {' → '.join(mq['types'])}\n"
        f"- **Event Types of the Wrong Match ({len(mc['types'])})**: {' → '.join(mc['types'])}\n"
        f"- **Event Triggers of the Query:** {', '.join(mq['triggers'])}\n"
        f"- **Event Triggers of the Wrong Match**: {', '.join(mc['triggers'])}\n"
        f"- **Text of the Query**: {cut(mq['text'])}\n"
        f"- **Text of the Wrong Match**: {cut(mc['text'])}\n"
    )
display(Markdown("\n".join(md)))

## Top 3 structural confusions under `qwen3_emb_0p6b__events_only` (by event-type order overlap)

### 1. `5504130__fr` → `562586__fr` &nbsp; (cosine similarity 0.797, event type order overlap 0.727, event type inventory 0.727, event trigger inventory overlap 0.182)

- **shared event-type skeleton (4 types)**: Come_together → Departing → Becoming → Know
- **shared genres**: (none) &nbsp;|&nbsp; langs: fr vs fr

- **Event Types of the Query (5)**: Come_together → Departing → Becoming → Giving → Know
- **Event Types of the Wrong Match (6)**: Destroying → Come_together → Departing → Becoming → Know → GetReady
- **Event Triggers of the Query:** meets, leave, became, gave, find
- **Event Triggers of the Wrong Match**: decimated, meets, left, turn, discover, prepares
- **Text of the Query**: A young man, Paul Harrison, the son of a wealthy British businessman living in Paris, meets a beautiful young orphan, Michelle Latour. The two teenagers leave Paris for the Camargue. Michelle became pregnant and gave birth to a baby girl. The young couple and the child lead a family life until the police find them.
- **Text of the Wrong Match**: An army veteran, the sole survivor of a decimated American patrol, meets a young South Korean, Short Round, as well as others left behind by the war. He leads them to an unoccupied Buddhist temple, which they turn into an observation camp. But when they discover that they are in the immediate vicinity of a North Korean communist camp, the troop prepares for the possibility of a fight...

### 2. `76940500__en` → `1029697__fr` &nbsp; (cosine similarity 0.811, event type order overlap 0.667, event type inventory 0.667, event trigger inventory overlap 0.000)

- **shared event-type skeleton (4 types)**: Sending → Arriving → Giving → Warning
- **shared genres**: (none) &nbsp;|&nbsp; langs: en vs fr

- **Event Types of the Query (6)**: Know → Sending → Releasing → Arriving → Giving → Warning
- **Event Types of the Wrong Match (6)**: Sending → Arriving → Giving → Warning → Placing → Becoming
- **Event Triggers of the Query:** found, sent, release, return, giving, threatened
- **Event Triggers of the Wrong Match**: transferred, arrives, gives, warning, placed, becomes
- **Text of the Query**: Doctor Gerda Maurer is found guilty of negligence and sent to prison. On her release she cannot return to her career in medicine and has to take on other jobs such as a nightclub singer. She is giving a second chance by a respected surgeon and works as his assistant, but her new position is threatened by the return of her former lover Georg.
- **Text of the Wrong Match**: Commissioner Betti is being transferred to Naples on orders from his hierarchy. As soon as he arrives on the scene, the local mafia gives him a threatening welcome in the form of a warning. The latter is placed under the  di 'O Generale  cup. At his instigation, the organization becomes the scene of a fratricidal reckoning.

### 3. `1029697__fr` → `76940500__en` &nbsp; (cosine similarity 0.811, event type order overlap 0.667, event type inventory 0.667, event trigger inventory overlap 0.000)

- **shared event-type skeleton (4 types)**: Sending → Arriving → Giving → Warning
- **shared genres**: (none) &nbsp;|&nbsp; langs: fr vs en

- **Event Types of the Query (6)**: Sending → Arriving → Giving → Warning → Placing → Becoming
- **Event Types of the Wrong Match (6)**: Know → Sending → Releasing → Arriving → Giving → Warning
- **Event Triggers of the Query:** transferred, arrives, gives, warning, placed, becomes
- **Event Triggers of the Wrong Match**: found, sent, release, return, giving, threatened
- **Text of the Query**: Commissioner Betti is being transferred to Naples on orders from his hierarchy. As soon as he arrives on the scene, the local mafia gives him a threatening welcome in the form of a warning. The latter is placed under the  di 'O Generale  cup. At his instigation, the organization becomes the scene of a fratricidal reckoning.
- **Text of the Wrong Match**: Doctor Gerda Maurer is found guilty of negligence and sent to prison. On her release she cannot return to her career in medicine and has to take on other jobs such as a nightclub singer. She is giving a second chance by a respected surgeon and works as his assistant, but her new position is threatened by the return of her former lover Georg.


### 1.4 Robustness across conditions and encoders
**Goal:** Repeat the error-vs-random comparison for every encoder × structural condition, to show the structural-confusion effect is a property of the event-based representation, not of one condition or encoder.

In [5]:
# Same error-vs-random construction as 1.1, repeated for every encoder × structural
# condition. Per cell of the grid the error set and the wrong matches change (different
# similarity matrix); the overlap measures and the per-query random null are built
# identically (same SEED). Takes several minutes (LCS over ~9k pairs per grid cell).
ALL_VIEWS = ["qwen3_emb_0p6b", "e5_mistral"]
ALL_CONDS = ["events_only", "temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]

rows = []
for view in ALL_VIEWS:
    for cond in ALL_CONDS:
        sim_c   = np.load(SIMILARITIES)[f"{view}__{cond}"]
        rank1_c = sim_c.argmax(axis=1)
        pairs_c = [(int(q), int(rank1_c[q])) for q in np.flatnonzero(works[rank1_c] != works)]
        rng_c   = random.Random(SEED)
        null_c  = []
        for q, _ in pairs_c:
            j = rng_c.randrange(N)
            while j == q or works[j] == works[q]:
                j = rng_c.randrange(N)
            null_c.append((q, j))
        wrong_share = f"{len(pairs_c):,}/{N:,} ({len(pairs_c) / N:.1%})"
        for name, fn in MEASURES.items():
            e = np.array([fn(meta[q], meta[c]) for q, c in pairs_c])
            n = np.array([fn(meta[q], meta[c]) for q, c in null_c])
            _, p = mannwhitneyu(e, n, alternative="greater")
            rows.append({"Encoder": view, "Condition": cond, "Wrong@1": wrong_share, "Measure": name,
                         "Error pair mean":  f"{e.mean():.4f}",
                         "Random-pair mean": f"{n.mean():.4f}",
                         "Δ":                f"{e.mean() - n.mean():+.4f}",
                         "p":                "< .001" if p < .001 else f"{p:.3f}"})
display(pd.DataFrame(rows).set_index(["Encoder", "Condition", "Wrong@1", "Measure"]))
display(Markdown(
    "*Note.* Identical construction to the 1.1 table, for every encoder × representation condition: "
    "each grid cell has its own similarity matrix, hence its own error set and wrong matches, while "
    "the overlap measures and the per-query random null are built the same way. Near-identical rows "
    "across the full grid show the structural-confusion effect is a property of the event-based "
    "representation itself, not of any single condition or encoder."
))

Error pair mean Random-pair mean        Δ       p
Encoder        Condition                   Wrong@1             Measure                                                                       
qwen3_emb_0p6b events_only                 3,382/4,315 (78.4%) Event-type inventory overlap          0.4319           0.2295  +0.2024  < .001
                                                               Event-type order overlap              0.2010           0.1116  +0.0894  < .001
                                                               Trigger inventory overlap             0.0980           0.0364  +0.0616  < .001
                                                               Trigger order overlap                 0.0644           0.0259  +0.0385  < .001
               temporal                    3,427/4,315 (79.4%) Event-type inventory overlap          0.4319           0.2321  +0.1999  < .001
                                                               Event-type order overlap              0.2063           0.1137  +0.0927  < .001
                                                               Trigger inventory overlap             0.0979           0.0364  +0.0615  < .001
                                                               Trigger order overlap                 0.0652           0.0263  +0.0389  < .001
               causal                      3,459/4,315 (80.2%) Event-type inventory overlap          0.4244           0.2318  +0.1926  < .001
                                                               Event-type order overlap              0.2006           0.1127  +0.0879  < .001
                                                               Trigger inventory overlap             0.0956           0.0362  +0.0594  < .001
                                                               Trigger order overlap                 0.0640           0.0256  +0.0384  < .001
               temporal_causal_independent 3,421/4,315 (79.3%) Event-type inventory overlap          0.4304           0.2305  +0.1999  < .001
                                                               Event-type order overlap              0.2060           0.1120  +0.0940  < .001
                                                               Trigger inventory overlap             0.0975           0.0369  +0.0606  < .001
                                                               Trigger order overlap                 0.0660           0.0263  +0.0397  < .001
               temporal_causal_joint       3,437/4,315 (79.7%) Event-type inventory overlap          0.4270           0.2309  +0.1961  < .001
                                                               Event-type order overlap              0.2044           0.1126  +0.0917  < .001
                                                               Trigger inventory overlap             0.0973           0.0379  +0.0594  < .001
                                                               Trigger order overlap                 0.0656           0.0270  +0.0387  < .001
e5_mistral     events_only                 3,608/4,315 (83.6%) Event-type inventory overlap          0.3956           0.2356  +0.1600  < .001
                                                               Event-type order overlap              0.2112           0.1146  +0.0967  < .001
                                                               Trigger inventory overlap             0.0976           0.0388  +0.0589  < .001
                                                               Trigger order overlap                 0.0715           0.0272  +0.0443  < .001
               temporal                    4,005/4,315 (92.8%) Event-type inventory overlap          0.3869           0.2349  +0.1520  < .001
                                                               Event-type order overlap              0.1992           0.1129  +0.0863  < .001
                                                               Trigger inventory overlap             0.0868         

*Note.* Identical construction to the 1.1 table, for every encoder × representation condition: each grid cell has its own similarity matrix, hence its own error set and wrong matches, while the overlap measures and the per-query random null are built the same way. Near-identical rows across the full grid show the structural-confusion effect is a property of the event-based representation itself, not of any single condition or encoder.

### Analysis 2 — Structural mismatch: why does adding relations make retrieval worse?

**Goal:** Adding temporal/causal relations consistently *lowers* retrieval performance relative to events-only, so we find the mechanism: does the relation layer add discriminative signal, or does it just make all stories look more alike?

**What the relation layer actually adds.** Relations are linearized in the hybrid format, e.g. *(e2:left|Departing, BEFORE, e1:arrived|Arriving)* — endpoints carry trigger and type. So, relative to the events-only string, the relation block adds three things:
- (a) **Relation labels** (e.g., *BEFORE*, *CAUSE*, ...), drawn from a taxonomies from literature
- (b) **Repeated copies of event tokens** the *EVENTS:* block already contains: an event linked in 10 relations gets its trigger/type repeated 10 times, re-weighting events by how often Llama links them, not by narrative importance
- (c) **Actual relation pattern** (which event pairs connect) — the only genuinely new information, and only useful if it is not predictable from event order

**Method:**
1. **Collapse check:** Mean and SD of the pairwise cosine similarity across all summaries, per condition. If the mean rises and the SD shrinks as relations are added, the relation layer pushes all stories closer together and compresses the spread that retrieval depends on.
2. **Label diversity:** — The distribution of relation labels per condition. If one label (e.g. *BEFORE*) dominates, the labels cannot discriminate between stories.
3. **Link-pattern predictability:** — The share of relations connecting adjacent events (*e_i → e_(i+1)*). If most links are adjacent-pair chains, the link pattern is derivable from event order and adds nothing new.
4. **Casualties:** — Queries where events-only retrieved the right story at rank 1 but a relation condition put a wrong story on top; inspect whether the wrong match wins on relation-pattern similarity despite different events.

**Results**


### 2.1 Collapse check
**Goal:** Show whether each added relation layer pushes *all* summaries closer together in embedding space (mean pairwise similarity up, spread down), instead of separating related from unrelated ones.

In [6]:
# Mean / SD / min / max of pairwise cosine similarity per encoder × condition, straight from similarities.npz.
# Rising mean + shrinking SD as relations are added = the space collapses (sameness, not signal).
sims_npz = np.load(SIMILARITIES)
rows = []
for view in ["qwen3_emb_0p6b", "e5_mistral"]:
    for cond in ["events_only", "temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]:
        m   = sims_npz[f"{view}__{cond}"]
        off = m[np.isfinite(m)]            # diagonal is -inf (self-similarity masked)
        rows.append({"Encoder": view, "Condition": cond,
                     "Mean pairwise cosine": f"{off.mean():.4f}",
                     "SD": f"{off.std():.4f}",
                     "Min": f"{off.min():.4f}",
                     "Max": f"{off.max():.4f}"})
display(pd.DataFrame(rows).set_index(["Encoder", "Condition"]))

Mean pairwise cosine      SD
Encoder        Condition                                               
qwen3_emb_0p6b events_only                               0.6183  0.0941
               temporal                                  0.6459  0.0897
               causal                                    0.6502  0.0889
               temporal_causal_independent               0.6588  0.0844
               temporal_causal_joint                     0.6695  0.0801
e5_mistral     events_only                               0.8890  0.0279
               temporal                                  0.9070  0.0378
               causal                                    0.9140  0.0281
               temporal_causal_independent               0.9237  0.0416
               temporal_causal_joint                     0.9169  0.0315

### 2.2 Relation-label diversity
**Goal:** Check whether the relation labels are diverse enough to distinguish stories, or whether one generic label (e.g. *BEFORE*) dominates everything.

In [ ]:
# Relation-label distribution per condition. Relations live in experiment.jsonl (4.4 GB),
# so they are streamed once into their own light cache, like the metadata in Analysis 1.
from collections import Counter

REL_CACHE   = EXPERIMENT_DIR / "error_analysis_relations.jsonl"
LLAMA_CONDS = ["temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]

if not REL_CACHE.exists():
    with open(EXPERIMENT, encoding="utf-8") as f, open(REL_CACHE, "w", encoding="utf-8") as out:
        for line in f:
            r = json.loads(line)
            row = {"wikidata_id": r["wikidata_id"], "summary_id": r["summary_id"], "conditions": {}}
            for cond in LLAMA_CONDS:
                rel = (r.get("conditions", {}).get(cond) or {}).get("relations") or {}
                row["conditions"][cond] = [[t["source"], t["relation"], t["target"]]
                                           for trs in rel.values() for t in (trs or [])]
            out.write(json.dumps(row) + "\n")
rels = [json.loads(l) for l in REL_CACHE.read_text(encoding="utf-8").splitlines() if l.strip()]
assert len(rels) == N, f"relations cache has {len(rels)} rows vs meta {N} — delete {REL_CACHE} and re-run"

rows = []
for cond in LLAMA_CONDS:
    cnt   = Counter(lab for row in rels for _, lab, _ in row["conditions"][cond])
    total = sum(cnt.values())
    for lab, c in cnt.most_common():
        rows.append({"Condition": cond, "Label": lab, "Count": c, "Share": f"{c/total:.1%}"})
display(pd.DataFrame(rows).set_index(["Condition", "Label"]))

### 2.3 Link-pattern predictability
**Goal:** Check whether the relations mostly link adjacent events — if so, the link pattern is derivable from event order and carries almost no new information.

In [ ]:
# Share of relations that just link adjacent events (e_i -> e_(i+1)), per condition.
# Uses the relations cache built in 2.2.
rows = []
for cond in LLAMA_CONDS:
    adj = fwd = total = 0
    for row in rels:
        for s, lab, t in row["conditions"][cond]:
            si, ti = int(s[1:]), int(t[1:])
            total += 1
            fwd   += si < ti
            adj   += abs(ti - si) == 1
    rows.append({"Condition": cond, "n relations": total,
                 "% adjacent (|target − source| = 1)": f"{adj/total:.1%}",
                 "% forward (source < target)":        f"{fwd/total:.1%}"})
display(pd.DataFrame(rows).set_index("Condition"))

### 2.4 Casualties (right → wrong)
**Goal:** Count the queries that events-only retrieved correctly but a relation condition broke — the direct damage the relation layer causes.

In [ ]:
# Per relation condition: how many queries events_only answered correctly at rank 1 become
# wrong ("broken"), how many wrong ones become right ("fixed"), and the net effect.
sims_npz   = np.load(SIMILARITIES)
BASE       = "events_only"
base_rank1 = sims_npz[f"{VIEW}__{BASE}"].argmax(axis=1)
base_right = works[base_rank1] == works

REL_CONDS = ["temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]
rows, broken_examples = [], {}
for cond in REL_CONDS:
    r1    = sims_npz[f"{VIEW}__{cond}"].argmax(axis=1)
    broke = np.flatnonzero(base_right  & (works[r1] != works))
    fixed = np.flatnonzero(~base_right & (works[r1] == works))
    broken_examples[cond] = [(int(q), int(base_rank1[q]), int(r1[q])) for q in broke]
    rows.append({"Condition": cond,
                 "Broken (right → wrong)": len(broke),
                 "Fixed (wrong → right)":  len(fixed),
                 "Net": len(fixed) - len(broke)})
display(pd.DataFrame(rows).set_index("Condition"))
display(Markdown(
    f"*Note.* The reference condition is **{BASE}** (encoder **{VIEW}**), which answers "
    f"{int(base_right.sum())} of the {N} queries correctly at rank 1. Each relation condition is "
    f"compared against it query by query: **Broken** counts queries {BASE} answered correctly whose "
    f"rank-1 retrieval becomes a wrong work once relations are added; **Fixed** counts queries {BASE} "
    f"got wrong that the relation condition repairs; **Net** = Fixed − Broken, so a negative value "
    f"means the relation layer breaks more retrievals than it fixes. Net / {N} equals the ΔP@1 of "
    f"Table 4 exactly — this table decomposes that aggregate delta into its gross movements: the "
    f"churn (Broken + Fixed) is several times larger than the net, i.e. relations substantially "
    f"reshuffle which stories are retrieved while slightly tilting the balance toward harm."
))

# A few broken examples: the correct story events_only found vs the wrong story the relation
# condition preferred, with event-type overlap for context.
EXAMPLE_COND, N_EXAMPLES = "temporal_causal_joint", 3
md = [f"#### Broken examples under `{VIEW}__{EXAMPLE_COND}` (events_only was right)\n"]
for q, good, bad in broken_examples[EXAMPLE_COND][:N_EXAMPLES]:
    mq, mg, mb = meta[q], meta[good], meta[bad]
    md.append(
        f"- query `{mq['wikidata_id']}__{mq['summary_id']}`: events_only → correct "
        f"`{mg['wikidata_id']}__{mg['summary_id']}` (type inventory {dice(mq['types'], mg['types']):.2f}); "
        f"{EXAMPLE_COND} → wrong `{mb['wikidata_id']}__{mb['summary_id']}` "
        f"(type inventory {dice(mq['types'], mb['types']):.2f})"
    )
display(Markdown("\n".join(md)))

### Analysis 3 — Annotation noise budget

**Goal:** Quantify everything the pipeline removed or silently lost between the raw subset (9,385 summaries) and the evaluated corpus (5,666), so the surviving noise is bounded, citable, and cannot silently explain the retrieval results.

**Method:**
1. **Pipeline funnel:** how many summaries each stage removed, and why.
2. **Llama failure modes:** split the §4.5 drop into JSON parse errors vs context-window overflows, plus rows that hit the generation token cap but still parsed (silently truncated relation lists).
3. **Silently dropped labels:** schema-valid relations whose label is outside the codebook (e.g. *AFTER*), dropped at validation — counted per condition and per source language (UNI-76).
4. **Hallucinated event IDs:** relations pointing at non-existent events, dropped in §4.6 (UNI-60) — rate and the *e(N+1)* pattern.
5. **Relation density & truncation:** relations per event, and whether token-cap rows have thinner relation lists.
6. **Residual trigger noise:** linking-verb triggers (~1%, retained) and subword fragments (fixed, expect ≈ 0).

**Results**


### 3.1 Pipeline funnel
**Goal:** One table of how many summaries each pipeline stage removed, and why, from the Section 2 Subseting down to the fully evaluated corpus

In [ ]:
# Summary counts per pipeline stage, with each stage's drop split by cause. Everything is
# recomputed from the stage files themselves (§5.5 from experiment.yaml). The §4.5 split
# needs the per-summary Llama outcome, so the llama_runs noise cache is built HERE (one
# streaming pass over llama_runs/<cond>.jsonl; 3.2–3.5 reuse it).
import re
import yaml

MIN_EVENTS, MIN_PER_WORK = 5, 2   # pipeline §3.5: MIN_EVENTS_FOR_RELATIONS, MIN_DEDUPED_EN_SUMMARIES

NOISE_CACHE = EXPERIMENT_DIR / "error_analysis_llama_noise.jsonl"
RAW_CONDS   = ["temporal", "causal", "temporal_causal_joint"]
RAW_KEY     = {"temporal": "temporal_relations", "causal": "causal_relations",
               "temporal_causal_joint": "joint_relations"}
ALLOWED = {c: set(yaml.safe_load((PROMPTS_DIR / f"{c}.yaml").read_text())["allowed_labels"])
           for c in RAW_CONDS}
EID_RE  = re.compile(r"^e\d+$")

# The stored `relations` are the VALIDATED triples, so dropped ones are recovered by
# re-parsing `response_raw` (kept verbatim as the audit trail — models/llama/inference.py).
if not NOISE_CACHE.exists():
    with open(NOISE_CACHE, "w", encoding="utf-8") as out:
        for cond in RAW_CONDS:
            with open(LLAMA_RUNS / f"{cond}.jsonl", encoding="utf-8") as f:
                for line in f:
                    r = json.loads(line)
                    b = r["condition_block"]
                    rec = {"condition": cond, "wikidata_id": r["wikidata_id"],
                           "summary_id": r["summary_id"], "lang": r.get("lang"),
                           "n_events": len(r.get("events") or []),
                           "overflow": b.get("source") == "skipped_ctx_overflow",
                           "parse_error": b.get("parse_error") is not None,
                           "hit_ctx_cap": bool(b.get("hit_ctx_cap")),
                           "n_kept": len((b.get("relations") or {}).get(RAW_KEY[cond]) or []),
                           "dropped_labels": {}, "n_drop_malformed": 0, "n_drop_eid": 0}
                    if not rec["overflow"] and not rec["parse_error"]:
                        try:
                            raw = json.loads(b.get("response_raw") or "")
                            triples = raw.get(RAW_KEY[cond]) if isinstance(raw, dict) else None
                        except Exception:
                            triples = None
                        bad_labels = Counter()
                        for t in triples or []:
                            if not isinstance(t, dict) or not all(k in t for k in ("relation", "source", "target")):
                                rec["n_drop_malformed"] += 1
                            elif t["relation"] not in ALLOWED[cond]:
                                bad_labels[str(t["relation"])] += 1
                            elif not (isinstance(t["source"], str) and EID_RE.match(t["source"])
                                      and isinstance(t["target"], str) and EID_RE.match(t["target"])):
                                rec["n_drop_eid"] += 1
                        rec["dropped_labels"] = dict(bad_labels)
                    out.write(json.dumps(rec, ensure_ascii=False) + "\n")
noise = [json.loads(l) for l in open(NOISE_CACHE, encoding="utf-8")]

# --- Section 3.5 split: < MIN_EVENTS events vs cluster floor (from the §3 output file) ---
n_full, n_few, work_counts = 0, 0, Counter()
for line in open(EXPERIMENT_DIR / "tma_subset_events_full.test.jsonl", encoding="utf-8"):
    r = json.loads(line)
    n_full += 1
    if len(r.get("events") or []) >= MIN_EVENTS:
        work_counts[r["wikidata_id"]] += 1
    else:
        n_few += 1
floor35     = sum(c for c in work_counts.values() if c < MIN_PER_WORK)
n_processed = n_full - n_few - floor35
assert n_processed == sum(1 for _ in open(EXPERIMENT_DIR / "tma_subset_events_processed.test.jsonl", encoding="utf-8"))

# --- Section 4.5 split: Llama complete-case vs cluster floor (from the noise cache) ---
failed = {(x["wikidata_id"], x["summary_id"]) for x in noise if x["parse_error"] or x["overflow"]}
surv = Counter()
for x in noise:
    if x["condition"] == RAW_CONDS[0] and (x["wikidata_id"], x["summary_id"]) not in failed:
        surv[x["wikidata_id"]] += 1
floor45 = sum(c for c in surv.values() if c < MIN_PER_WORK)

# --- Section 5.5 split comes pre-counted from experiment.yaml ---
y = yaml.safe_load((EXPERIMENT_DIR / "experiment.yaml").read_text())["pre_embedding_e5_truncation_filter"]
assert n_processed - len(failed) - floor45 == y["summaries_in"]
# experiment.yaml records the PIPELINE output (§5.5 -> summaries_out). After it, pipeline §8.0
# collapses the corpus to the matched intersection I' (summaries surviving in BOTH arms + cluster
# floor); N reflects that final evaluated set. summaries_out == N only when §8.0 has not been run.
n_matched_drop = y["summaries_out"] - N
assert n_matched_drop >= 0, "experiment.jsonl is smaller-than-expected vs the §5.5 output — check N / experiment.yaml"

pct = lambda n, base: f"{n:,} ({n / base:.1%})"
funnel_rows = [
    {"Stage": "Section 2 subsetting (output)", "Cause": "official test split; genre, translation-dedup, length, BERT-subword, ≥2-summaries-per-work filters",
     "Dropped (% of stage input)": "—", "Summaries out": n_full},
    {"Stage": "Section 3.5 pre-annotation", "Cause": f"fewer than {MIN_EVENTS} detected events (structural-informativeness floor)",
     "Dropped (% of stage input)": pct(n_few, n_full), "Summaries out": ""},
    {"Stage": "Section 3.5 pre-annotation", "Cause": f"cluster floor (work left with < {MIN_PER_WORK} summaries)",
     "Dropped (% of stage input)": pct(floor35, n_full), "Summaries out": n_processed},
    {"Stage": "Section 4.5 complete-case", "Cause": "Llama relations missing in ≥1 condition (parse error / ctx overflow — split in 3.2)",
     "Dropped (% of stage input)": pct(len(failed), n_processed), "Summaries out": ""},
    {"Stage": "Section 4.5 complete-case", "Cause": f"cluster floor (work left with < {MIN_PER_WORK} summaries)",
     "Dropped (% of stage input)": pct(floor45, n_processed), "Summaries out": y["summaries_in"]},
    {"Stage": "Section 5.5 linearization cap", "Cause": f"linearized string over {y['cap_tokens']} tokens in any condition",
     "Dropped (% of stage input)": pct(y["summaries_dropped_overflow"], y["summaries_in"]), "Summaries out": ""},
    {"Stage": "Section 5.5 linearization cap", "Cause": f"cluster floor (work left with < {MIN_PER_WORK} summaries)",
     "Dropped (% of stage input)": pct(y["summaries_dropped_new_singletons"], y["summaries_in"]), "Summaries out": y["summaries_out"]},
]
# Post-pipeline matched-intersection restriction (§8.0) — shown only when it actually applied.
if n_matched_drop:
    funnel_rows.append(
        {"Stage": "Post-pipeline §8.0 matched intersection", "Cause": "kept only summaries surviving in BOTH the non-anon and anon arms (∩ + ≥2-summaries-per-work floor)",
         "Dropped (% of stage input)": pct(n_matched_drop, y["summaries_out"]), "Summaries out": N})
funnel = pd.DataFrame(funnel_rows)
display(funnel.set_index(["Stage", "Cause"]))
print(f"Retained end-to-end: {N:,} / {n_full:,} summaries ({N / n_full:.1%}); "
      f"percentages are relative to each stage's input.")

### 3.2 Llama failure modes
**Goal:** Split the Section 4.5 drop into its causes — JSON parse errors vs context-window overflows — and count the rows whose generation hit the token cap yet still parsed (silently truncated relation lists).

In [ ]:
# Failure modes per condition, from the noise cache built in 3.1.
rows = []
for cond in RAW_CONDS:
    sub = [x for x in noise if x["condition"] == cond]
    rows.append({"Condition": cond, "Summaries": len(sub),
                 "Parse errors": sum(x["parse_error"] for x in sub),
                 "Context overflows": sum(x["overflow"] for x in sub),
                 "Hit token cap but parsed": sum(x["hit_ctx_cap"] and not x["parse_error"] and not x["overflow"] for x in sub)})
display(pd.DataFrame(rows).set_index("Condition"))
display(Markdown(
    "*Note.* **Parse error** = Llama's output was not valid JSON (nothing salvaged for that "
    "condition). **Context overflow** = the input alone exceeded the context window, so no "
    "generation ran (`source = skipped_ctx_overflow`). A summary failing in *any* condition was "
    "dropped whole by the §4.5 complete-case filter — these columns are the cause split behind that "
    "funnel stage (their union across conditions is the complete-case count in 3.1). "
    "**Hit token cap but parsed** = generation reached `max_new_tokens` yet still produced valid "
    "JSON: the relation list may be silently incomplete (assessed in 3.5)."
))

### 3.3 Silently dropped relation labels
**Goal:** Count the schema-valid relations Llama emitted with labels outside the codebook (silently dropped at validation), and check whether the loss is systematic by the summary's source language (UNI-76).

In [ ]:
# Which out-of-codebook labels Llama invents (silently dropped at validation), per condition;
# then the drop rate per source language — a systematic per-language skew = UNI-76 confirmed.
rows = []
for cond in RAW_CONDS:
    agg = Counter()
    n_malformed = n_eid = 0
    for x in noise:
        if x["condition"] == cond:
            agg.update(x["dropped_labels"])
            n_malformed += x["n_drop_malformed"]
            n_eid       += x["n_drop_eid"]
    kept, dropped = sum(x["n_kept"] for x in noise if x["condition"] == cond), sum(agg.values())
    for lab, c in agg.most_common(8):
        rows.append({"Condition": cond, "Dropped as": f"invalid label `{lab}`", "Count": c})
    rows.append({"Condition": cond, "Dropped as": "malformed entry / bad eID format",
                 "Count": n_malformed + n_eid})
    rows.append({"Condition": cond, "Dropped as": "TOTAL invalid-label",
                 "Count": f"{dropped:,} ({dropped / (dropped + kept):.2%} of emitted)"})
display(pd.DataFrame(rows).set_index(["Condition", "Dropped as"]))

# Per source language (top 10 by row count): dropped / (kept + dropped)
top_langs = [l for l, _ in Counter(x["lang"] for x in noise).most_common(10)]
lang_rows = []
for lang in top_langs:
    row = {"Lang": lang, "n summaries": sum(x["condition"] == RAW_CONDS[0] and x["lang"] == lang for x in noise)}
    for cond in RAW_CONDS:
        sub  = [x for x in noise if x["condition"] == cond and x["lang"] == lang]
        kept = sum(x["n_kept"] for x in sub)
        drp  = sum(sum(x["dropped_labels"].values()) for x in sub)
        row[cond] = f"{drp / (kept + drp):.2%}" if (kept + drp) else "—"
    lang_rows.append(row)
display(pd.DataFrame(lang_rows).set_index("Lang"))
display(Markdown(
    "*Note.* A dropped label is a schema-valid relation whose label is not in the condition's "
    "codebook (e.g. `AFTER` when only `BEFORE/OVERLAPS/CONTAINS/IDENTITY` are allowed) — "
    "`parse_and_validate` drops it silently and keeps the rest of the row; `response_raw` is the "
    "audit trail it is recovered from here. The per-language table tests UNI-76: all inputs are "
    "English text, so a higher drop rate for some source languages means the *style* of "
    "machine-translated summaries systematically steers Llama's label vocabulary, leaving those "
    "summaries with thinner relation sets."
))

### 3.4 Hallucinated event IDs
**Goal:** Report the rate of relations pointing at non-existent event IDs (dropped in §4.6) and the *e(N+1)* pattern behind them (UNI-60).

In [ ]:
# Hallucinated event IDs (dropped in pipeline §4.6, logged to hallucinated_relations.jsonl).
# Kept-relation denominators come from the 2.2 relations cache (final 5,666-summary corpus);
# hallucinated triples were logged one stage earlier (§4.6, 6,189 summaries), so rates are
# slightly conservative approximations.
hall = [json.loads(l) for l in open(EXPERIMENT_DIR / "hallucinated_relations.jsonl", encoding="utf-8")]

rows = []
for cond in ["temporal", "causal", "temporal_causal_joint", "temporal_causal_independent"]:
    h    = [x for x in hall if x["condition"] == cond]
    kept = sum(len(row["conditions"][cond]) for row in rels)
    nseq = sum(bool(x.get("is_next_in_sequence")) for x in h)
    rows.append({"Condition": cond, "Hallucinated": len(h), "Kept": kept,
                 "Rate": f"{len(h) / (len(h) + kept):.2%}",
                 "% exactly e(N+1)": f"{nseq / len(h):.1%}" if h else "—"})
display(pd.DataFrame(rows).set_index("Condition"))
display(Markdown(
    f"*Note.* A hallucinated relation references an event ID that does not exist in the summary "
    f"(e.g. `e79` when only `e1..e78` were given). All {len(hall):,} were dropped before "
    f"linearization (§4.6). **{sum(bool(x.get('is_next_in_sequence')) for x in hall) / len(hall):.1%} "
    f"invent exactly the next ID after the last real event** — the model assumes the story must keep "
    f"connecting and fabricates a successor for the final event (UNI-60). "
    f"`temporal_causal_independent` is composed post-hoc from the temporal + causal runs, so its "
    f"row is the union of those two, not a separate LLM run."
))